# Differential expression analysis


You have run the nf-core/rnaseq pipeline and checked the first quality control metrics of your fastq files. This was, however, only the primary analysis and we want to take it further.

Due to the computational demand of the pipeline, you only ran the pipeline on two of the 16 samples in the study yesterday. We provide you an essential output of nf-core/rnaseq pipeline in the `data` folder: It contains the combined epression matrix as produced by Salmon, which provides transcript levels for each gene (rows) and each sample (columns).


## Task 1
We would now like to understand exactly the difference between the expression in our groups of mice. 
Which pipeline would you use for this?

Differential expression in our groups of mice can be done using the pipeline **nf-core/differentialabundance**.

Have a close look at the pipeline's "Usage" page on the [nf-core docs](nf-co.re). You will need to create a samplesheet (based on the column names in the provided matrix).

Please paste here the command you used. You may need to inspect the provided expression matrix more closely and create additional files, like a samplesheet (based on the column names) or a contrast file (there happens to also be one in `data/` that you can use).

In [ ]:
# TODO instead of the genome we can also provide the lenghts directly (day_03/data/rnaseq-out-small/star_salmon/salmon.merged.gene_lengths.tsv)
!nextflow run nf-core/differentialabundance \
    -r 2.0.0 \
    -profile docker \
    --outdir diff_abundance_out \
    --input samplesheet_day03.csv \
    --contrasts data/contrasts.csv \
    --study_name mouse_oxy_study \
    --study_type rnaseq \
    --genome GRCm38 \
    --matrix data/salmon.merged.gene_counts.tsv 

In [ ]:
# bash script for getting the just the samples from the header line 
!head -1 data/salmon.merged.gene_counts.tsv | sed "s/\t/\n/g" | tail -n +3

Sham_oxy_1
Sham_oxy_2
Sham_oxy_3
Sham_oxy_4
Sham_Sal_1
Sham_Sal_2
Sham_Sal_3
Sham_Sal_4
SNI_oxy_1
SNI_oxy_2
SNI_oxy_3
SNI_oxy_4
SNI_Sal_1
SNI_Sal_2
SNI_Sal_3
SNI_Sal_4


In [ ]:
# code for creating the samplesheet

import pandas as pd
import re

samplesheet_df = pd.DataFrame()

# TODO add bash script above into a subprocess inside this python script

# samples from the salmon.merged.gene_counts.tsv (RNAseq output data)
samples = [
    "Sham_oxy_1",
    "Sham_oxy_2",
    "Sham_oxy_3",
    "Sham_oxy_4",
    "Sham_Sal_1",
    "Sham_Sal_2",
    "Sham_Sal_3",
    "Sham_Sal_4",
    "SNI_oxy_1",
    "SNI_oxy_2",
    "SNI_oxy_3",
    "SNI_oxy_4",
    "SNI_Sal_1",
	"SNI_Sal_2",
    "SNI_Sal_3",
    "SNI_Sal_4",
]

# add the samples under the header sample (according to the usage page on nf-core)
samplesheet_df["sample"] = samples

# get the condition of each sample (sample string with the replicate trimmed using regex)
conditions = [re.sub("\\_\\d*$", '', sample) for sample in samples]
# Important: condition has to be written with a capital C in the header in order to match the "Condition" in the contrasts file 
samplesheet_df["Condition"] = conditions

print(samplesheet_df)

samplesheet_df.to_csv("samplesheet_day03.csv", index=False)

        sample Condition
0   Sham_oxy_1  Sham_oxy
1   Sham_oxy_2  Sham_oxy
2   Sham_oxy_3  Sham_oxy
3   Sham_oxy_4  Sham_oxy
4   Sham_Sal_1  Sham_Sal
5   Sham_Sal_2  Sham_Sal
6   Sham_Sal_3  Sham_Sal
7   Sham_Sal_4  Sham_Sal
8    SNI_oxy_1   SNI_oxy
9    SNI_oxy_2   SNI_oxy
10   SNI_oxy_3   SNI_oxy
11   SNI_oxy_4   SNI_oxy
12   SNI_Sal_1   SNI_Sal
13   SNI_Sal_2   SNI_Sal
14   SNI_Sal_3   SNI_Sal
15   SNI_Sal_4   SNI_Sal


Explain all the parameters you set and why you set them in this way. If you used or created additional files as input, explain what they are used for.

**Parameters:**
- **-r 2.0.0**: sets the version of the differentialabundance pipeline
- **-profile docker**: selects docker as the profile so that the pipeline is run in a container
- **--outdir diff_abundance_out**: selects the directory where output is put
- **--input samplesheet_day03.csv**: selects the samplesheet CSV that I created in python
- **--constrasts data/contrasts.csv**: selects the contrasts CSV from the provided data directory as the contrasts file (used to define groups of samples from the 'input' file for the comparisons)
- **--study_name mouse_oxy_study**: *TODO*
- **--study_type rnaseq**: *TODO*
- **--matrix day_03/data/salmon.merged.gene_counts.tsv**: *TODO*
- **--genome GRCm38**: selects the mouse genome; alternative to the gene length file

**Additional files:**
- samplesheet CSV for identification of the samples' condition
    - no fastq_1 or fastq_2 files because they are not necessary for the differential abundances pipeline
- contrasts CSV from the provided data directory for defining groups of samples from the 'input' file for the comparison (conditions match the samplesheet's conditions)


What were the outputs of the pipeline?

**The output directory contains:**
- pipeline_info directory
    - contains HTML files for the report and the processes execution timeline
- plots directory
    - differential, exploratory and qc plots
- report directory
    - HTML file with the pipeline results in the following schema: abstract, data, results, methods, appendices 
- shinyngs_app directory
    - *TODO*
- tables directory
    - differential tables
    - processed abundance tables

In [2]:
#!TODO

Would you exclude any samples? If yes, which and why?

At first glance, the plot provided in the outlier detection section of the pipeline report reports no outliers in groups defined by condition (see [Fig. 1](outlier_detection.png)), however looking at the sample clustering dendrogram (see [Fig. 2](diff_abundance_out/plots/exploratory/contrasts/Condition/png/sample_dendrogram.png)) the samples SNI_Sal_2 and SNI_Sal_4 cluster far together far away from all other samples forming the outgroup of the dendrogram which indicates that they may be outliers.

These outliers should be excluded since they're likely erroneous otherwise they would not be as different from the other samples. Not excluding them is a risk for the following analyses


![outlier](outlier_detection.png)

Fig. 1: Outlier detection plot


![dendro](diff_abundance_out/plots/exploratory/contrasts/Condition/png/sample_dendrogram.png)

Fig. 2: Sample clustering dendrogram

How many genes were differentially expressed in each contrast? Does this confirm what the paper mentions?

In [27]:
# TODO code for creating the contrasts.csv with the comparisons mentioned in the paper
import csv

with open("data/contrasts.csv", "r") as file:
    reader = csv.reader(file)
    header = next(reader)   

print(header)

contrasts = [("Sham_Sal", "Sham_oxy"),("SNI_Sal","SNI_oxy"),("SNI_oxy","Sham_Sal")]

with open("contrasts_edited.csv", "w") as file:
    # write the header
    for i, col_name in enumerate(header):    
        file.write(col_name)
        if i != len(header)-1:
            file.write(",")
        else: 
            file.write("\n")
    # write the contrasts
    for contrast in contrasts:
        file.write(f"{contrast[0]}_vs_{contrast[1]}" + "," + "Condition," + contrast[0] + "," + contrast[1]+ "\n")




['id', 'variable', 'reference', 'target']


**Differentially expressed genes according to the pipeline report:**
*TODO change the contrast file for this comparison*

- Sham-Oxy vs. Sham-Sal:
- SNI-Oxy vs. SNI-Sal:
- SNI-Oxy vs. Sham-Sal:

**Differentially expressed genes according to the paper:**

- Sham-Oxy vs. Sham-Sal:
- SNI-Oxy vs. SNI-Sal:
- SNI-Oxy vs. Sham-Sal:


The paper mentions differentially expressed genes in three brain regions : the NAc, mPFC and VTA. Briefly explain what these 3 regions are.

**NAc**:

NAc refers to the nucleus accumbens. The nucleus accumbens is a brain region in the ventral striatum that is involved in the cognitive processing of motivation, aversion, reward and reinforcement learning and thus plays an important role in addiction.

**mPFC**:

mPFC refers to the medial prefrontal cortex. It is part of the prefrontal cortex and is involved in multiple higher-order functions such as working memory and risk processing.


**VTA**:

VTA refers to the ventral tegmental area which is located close to the floor of the midbrain. It is involved in numerous processes such as reward cognition (motivational salience, associative learning, and positively-valenced emotions) and orgasm.


Is there anyway from the paper and the material and methods for us to know which genes are included in these regions?

Once you have your list of differentially expressed genes, do you think just communicating those to the biologists would be sufficient? What does the publication state?

Please reproduce the Venn Diagram from Figure 3, not taking into account the brain regions but just the contrasts mentionned.